In [0]:
df_taxa_bruta_nat_bronze = spark.read.table("workspace.bronze.taxa_bruta_natalidade")
display(df_taxa_bruta_nat_bronze.limit(10))

In [0]:
df_esp_vida_nasc_bronze = spark.read.table("workspace.bronze.esperanca_vida_nascer")
display(df_esp_vida_nasc_bronze.limit(10))

### Pradonizando "None" (está como String) para NULL

In [0]:
from pyspark.sql import functions as F

colunas_tratamento_string = ["no_municipio", "no_regiao_saude", "no_macro"]
colunas_tratamento_conversao = ["vl_indicador_calculado_mun", "vl_indicador_calculado_rs", "vl_indicador_calculado_ms", "vl_indicador_calculado_al"]
colunas_tratamento_inteiros = ["co_ibge", "co_regiao_saude", "co_macro"]

# def limpeza_conversao(df, colunas_tratamento_string, colunas_tratamento_conversao):
#     for coluna in colunas_tratamento_string:
#         df = df.withColumn(
#             coluna,
#             F.when(F.col(coluna) == "None", None)
#             .otherwise(F.col(coluna))
#             .cast("string")
#         )
#     for coluna in colunas_tratamento_conversao:
#         df = df.withColumn(
#             coluna,
#             F.when(F.col(coluna) == "None", None)
#             .otherwise(F.col(coluna))
#             .cast("decimal(10,2)")
#         )
#     return df

# Função para limpeza e conversão de tipos
def limpeza_conversao(df, colunas_tratamento_string, colunas_tratamento_conversao, colunas_tratamento_inteiros):
    return df.select(
        *[
            F.when(F.col(coluna).isin("None", "null", "NA", "nan"), None).otherwise(F.col(coluna)).cast("string").alias(coluna)
            if coluna in colunas_tratamento_string else
            F.when(F.col(coluna).isin("None", "null", "NA", "nan"), None).otherwise(F.col(coluna)).cast("decimal(10,2)").alias(coluna)
            if coluna in colunas_tratamento_conversao else
            F.when(F.col(coluna).isin("None", "null", "NA", "nan"), None).otherwise(F.col(coluna)).cast("int").alias(coluna)
            if coluna in colunas_tratamento_inteiros else
            F.col(coluna)
            for coluna in df.columns
        ]
    )

df_taxa_bruta_nat_silver = limpeza_conversao(df_taxa_bruta_nat_bronze, colunas_tratamento_string, colunas_tratamento_conversao, colunas_tratamento_inteiros)
df_esp_vida_nasc_silver = limpeza_conversao(df_esp_vida_nasc_bronze, colunas_tratamento_string, colunas_tratamento_conversao, colunas_tratamento_inteiros)    

### Seperando a coluna co_anomes em duas novas colunas

In [0]:
# Função para separar os anos e meses
def separa_coanomes(df, coluna="co_anomes"):
    return(
        df.withColumn(
            "co_ano",
            F.substring(F.col(coluna).cast("string"), 1, 4).cast("int")
        )
        .withColumn(
            "co_mes",
            F.substring(F.col(coluna).cast("string"), 5, 2).cast("int")
        )
        .drop(coluna)
    )

df_taxa_bruta_nat_silver = separa_coanomes(df_taxa_bruta_nat_silver)
df_esp_vida_nasc_silver = separa_coanomes(df_esp_vida_nasc_silver)

### Limpando o Timestamp

In [0]:
# Função para separar as datas
def separa_datas(df):
    return(
        df
        .withColumn("dt_competencia", F.to_date("dt_competencia"))
        .withColumn("dt_atualizacao", F.to_date("dt_atualizacao"))
    )

df_taxa_bruta_nat_silver = separa_datas(df_taxa_bruta_nat_silver)
df_esp_vida_nasc_silver = separa_datas(df_esp_vida_nasc_silver)

### Ajustando casas decimais

**Essa função poderia ser evitada se eu adicionasse as colunas faltantes e modificado a função clean_cast_two(), porém achei melhor fazer ela separada.**

In [0]:
colunas_decimais = ["vl_indicador_calculado_mun", "vl_indicador_calculado_rs", "vl_indicador_calculado_ms", "vl_indicador_calculado_uf", "vl_indicador_calculado_reg", "vl_indicador_calculado_br"]

# Função para converter colunas em decimal
def tipo_decimal(df, colunas_decimais):
    return df.select(
        *[F.col(coluna).cast("decimal(10,2)").alias(coluna) if coluna in colunas_decimais else F.col(coluna) for coluna in df.columns]
    )

df_taxa_bruta_nat_silver = tipo_decimal(df_taxa_bruta_nat_silver, colunas_decimais)
df_esp_vida_nasc_silver = tipo_decimal(df_esp_vida_nasc_silver, colunas_decimais)

### Indicando tabela de origem

In [0]:
# Função para adicionar uma coluna com o nome da tabela de origem do dado
def col_origin(df, nome_origem):
    return df.withColumn("col_ds_origem", F.lit(nome_origem))

df_taxa_bruta_nat_silver = col_origin(df_taxa_bruta_nat_silver, "taxa_bruta_natalidade")
df_esp_vida_nasc_silver = col_origin(df_esp_vida_nasc_silver, "esperanca_vida_nascer")

### Limpando informações

- **sg_categoria e ds_categoria: códigos internos do DATASUS.**
- **co_item_categoria: é sempre 0.**
- **sg_granularidade e ds_granularidade: sempre "UF/Unidade Federativa".**

In [0]:
df_esp_vida_nasc_silver = df_esp_vida_nasc_silver.drop("sg_categoria", "ds_categoria", "co_item_categoria", "sg_granularidade", "ds_granularidade")
df_taxa_bruta_nat_silver = df_taxa_bruta_nat_silver.drop("sg_categoria", "ds_categoria", "co_item_categoria", "sg_granularidade", "ds_granularidade")

In [0]:
display(df_taxa_bruta_nat_silver.limit(10))

In [0]:
display(df_esp_vida_nasc_silver.limit(10))

### Salvando a Silver no Delta Lake

In [0]:
%sql
-- criando um schema silver para salvar os dados em Delta
CREATE SCHEMA IF NOT EXISTS workspace.silver;
USE SCHEMA silver;

In [0]:
df_taxa_bruta_nat_silver.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "True") \
.saveAsTable("workspace.silver.taxa_bruta_natalidade")

df_esp_vida_nasc_silver.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "True") \
.saveAsTable("workspace.silver.esperanca_vida_nascer")

print("Tabelas (silver) criadas corretamente.")